# Four points on a line

$$
\begin{aligned}
x &: 0, 1, 2, 3 \\
y &: -1, -1, +1, +1
\end{aligned}
$$

The classes are separated by a gap from $x = 1$ to $x = 2$. This lesson first solves the hard margin, then turns $C$ down and watches the same points again.


## Hard margin

The nearest points are $x = 1$ and $x = 2$. The gap has length $1$, so each side of the street has width $1/2$. The center is $x = 3/2$.

Functional margin $1$ at those two points fixes the scale:

$$
\begin{aligned}
-(w + b) &= 1 \\
2w + b &= 1
\end{aligned}
$$

Add them: $w = 2$. Then $b = -3$. The score is $f(x) = 2x - 3$.

| $x$ | $f(x)$ | functional margin |
|---:|---:|---:|
| $0$ | $-3$ | $3$ |
| $1$ | $-1$ | $1$ |
| $2$ | $1$ | $1$ |
| $3$ | $3$ | $3$ |

The multipliers sit only on the two edge points. Balance says they are equal, and $w = -\alpha_1\cdot 1 + \alpha_2\cdot 2$ with the outer multipliers at zero gives $\alpha = 2$ on each edge point:

$$
\boldsymbol{\alpha} = (0,\ 2,\ 2,\ 0)
$$

The primal cost is $\tfrac{1}{2}(2)^2 = 2$.


In [1]:
# Hard-margin check on the four-point line.
from fractions import Fraction

xs = [0, 1, 2, 3]
ys = [-1, -1, 1, 1]
w, b = Fraction(2), Fraction(-3)
margins = [y * (w * x + b) for x, y in zip(xs, ys)]
assert margins == [3, 1, 1, 3]
alpha = [Fraction(0), Fraction(2), Fraction(2), Fraction(0)]
recovered = sum(a * y * x for a, y, x in zip(alpha, ys, xs))
assert recovered == w
assert sum(a * y for a, y in zip(alpha, ys)) == 0
print("margins", margins)
print("hard cost", Fraction(1, 2) * w ** 2)


margins [Fraction(3, 1), Fraction(1, 1), Fraction(1, 1), Fraction(3, 1)]
hard cost 2


## Turn the price down to $C = 1$

The hard-margin multipliers are $2$, which is above $C = 1$, so the two edge multipliers clip to $1$. The outer multipliers stay $0$:

$$
\boldsymbol{\alpha} = (0,\ 1,\ 1,\ 0), \qquad w = -1\cdot 1 + 1\cdot 2 = 1
$$

With $w$ fixed, choose $b$ by looking at the four hinges:

$$
\begin{aligned}
\xi_0 &= \max(0,\ 1 + b) \\
\xi_1 &= \max(0,\ 2 + b) \\
\xi_2 &= \max(0,\ -1 - b) \\
\xi_3 &= \max(0,\ -2 - b)
\end{aligned}
$$

For every $b$ in $[-2, -1]$ the two outer hinges are $0$ and the two inner hinges add to $1$:

$$
(2 + b) + (-1 - b) = 1
$$

The $b$ terms cancel. Every cut between $x = 1$ and $x = 2$ has the same soft cost

$$
\frac{1}{2}(1)^2 + 1\cdot 1 = \frac{3}{2}
$$

The balanced cut is $b = -3/2$, so $f(x) = x - 3/2$. Then $\xi = (0,\ 1/2,\ 1/2,\ 0)$. Both inner points are correctly classified and inside the street.

At the endpoint $b = -1$, the cut sits on $x = 1$. That point has score $0$ and slack $1$. The cost is still $3/2$. A solver may return any bias in the interval. Check the cost, not a single lucky $b$.


In [2]:
# For C = 1 the whole bias interval has soft cost 3/2.
from fractions import Fraction

def hinges(b):
    return [
        max(Fraction(0), 1 + b),
        max(Fraction(0), 2 + b),
        max(Fraction(0), -1 - b),
        max(Fraction(0), -2 - b),
    ]

w = Fraction(1)
for b in (Fraction(-2), Fraction(-3, 2), Fraction(-1)):
    slack = sum(hinges(b))
    cost = Fraction(1, 2) * w ** 2 + slack
    assert slack == 1
    assert cost == Fraction(3, 2)
# Just outside either end, the cost rises.
assert sum(hinges(Fraction(-21, 10))) > 1
assert sum(hinges(Fraction(-9, 10))) > 1
balanced = hinges(Fraction(-3, 2))
assert balanced == [0, Fraction(1, 2), Fraction(1, 2), 0]
print("balanced slacks", balanced)


balanced slacks [Fraction(0, 1), Fraction(1, 2), Fraction(1, 2), Fraction(0, 1)]


## A large enough $C$ returns the hard margin

On this line the clipped weight is $w = C$ while $C < 2$, because each edge multiplier stops at $C$ and $w = -C + 2C$. Once $C \ge 2$, the box no longer cuts the hard-margin multipliers. The solution freezes:

$$
w = 2, \qquad b = -3, \qquad \boldsymbol{\alpha} = (0,\ 2,\ 2,\ 0), \qquad \boldsymbol{\xi} = \mathbf{0}
$$

At $C = 1.5$ you can see the freeze coming. Then $w = 1.5$, the balanced bias is $-2.25$, and each inner slack is $1/4$. Total slack is $1/2$. Soft cost:

$$
\frac{1}{2}\left(\frac{3}{2}\right)^2 + \frac{3}{2}\cdot\frac{1}{2}
= \frac{9}{8} + \frac{3}{4}
= \frac{15}{8}
$$


In [3]:
# C = 1.5 sits between the clipped line and the hard margin.
from fractions import Fraction

C = Fraction(3, 2)
w = C
b = Fraction(-9, 4)  # balanced cut: -(3/2) * (3/2)
xs = [0, 1, 2, 3]
ys = [-1, -1, 1, 1]
slacks = []
for x, y in zip(xs, ys):
    margin = y * (w * x + b)
    slacks.append(max(Fraction(0), 1 - margin))
assert slacks == [0, Fraction(1, 4), Fraction(1, 4), 0]
cost = Fraction(1, 2) * w ** 2 + C * sum(slacks)
assert cost == Fraction(15, 8)
# C = 3 does not move past the hard-margin weight.
assert min(Fraction(3), Fraction(2)) == 2
print("C = 1.5 slacks", slacks, "cost", cost)


C = 1.5 slacks [Fraction(0, 1), Fraction(1, 4), Fraction(1, 4), Fraction(0, 1)] cost 15/8


## A pitfall

Do not compare a hard objective with a soft objective and declare a winner. They charge different bills. Compare two candidates on the **same** objective. Also, if every multiplier is $0$ or $C$, do not expect one magic bias. Expect an interval, and say so.

## What to carry forward

On this line, $C \ge 2$ recovers $f(x) = 2x - 3$. At $C = 1$, $w = 1$ and every $b \in [-2, -1]$ costs $3/2$.
